# 🚀 AI Agent Platform Labs — Environment Setup

This notebook will guide you through deploying all Azure resources needed for Labs 01-07.

## What This Notebook Does

1. **Checks prerequisites** — Python version, Azure CLI, required tools
2. **Deploys Azure resources** — All services via Bicep (one command)
3. **Generates your `.env` file** — All connection strings in one place

**Estimated time:** ~20 minutes

---

## Step 1: Check Prerequisites

> ⚠️ **Important:** Make sure this notebook is using **Python 3.11+** as its kernel.
> Click the kernel version in the **top-right corner** of this notebook and select a Python 3.11+ environment.
> If you created a `.venv`, select that.

In [15]:
import sys
import subprocess
import shutil

print("🔍 Checking Prerequisites...")
print("=" * 50)

all_ok = True

# Check Python version (of the NOTEBOOK KERNEL, not system Python)
py_ver = sys.version_info
py_ok = py_ver >= (3, 11)
print(f"\n📌 Python (notebook kernel): {py_ver.major}.{py_ver.minor}.{py_ver.micro}")
if py_ok:
    print("   ✅ OK")
else:
    print("   ❌ This notebook is using Python < 3.11!")
    print()
    print("   🔧 HOW TO FIX:")
    print("   1. Click the Python version in the TOP-RIGHT corner of this notebook")
    print("   2. Select 'Python Environments'")
    print("   3. Choose Python 3.11+ (or your .venv if you created one)")
    print()
    # Try to find a newer Python on the system
    for ver in ["python3.13", "python3.12", "python3.11", "python3"]:
        path = shutil.which(ver)
        if path:
            try:
                result = subprocess.run([path, "--version"], capture_output=True, text=True)
                print(f"   💡 Found {result.stdout.strip()} at: {path}")
            except Exception:
                pass
all_ok = all_ok and py_ok

# Check Azure CLI
az_ok = shutil.which("az") is not None
print(f"\n📌 Azure CLI: {'✅ Found' if az_ok else '❌ Not found — install: https://aka.ms/installazurecli'}")
all_ok = all_ok and az_ok

# Check jq
jq_ok = shutil.which("jq") is not None
print(f"\n📌 jq: {'✅ Found' if jq_ok else '❌ Not found — install: brew install jq (macOS) or winget install jqlang.jq (Windows)'}")
all_ok = all_ok and jq_ok

# Check Git
git_ok = shutil.which("git") is not None
print(f"\n📌 Git: {'✅ Found' if git_ok else '❌ Not found'}")
all_ok = all_ok and git_ok

# Check Azure login
if az_ok:
    try:
        result = subprocess.run(["az", "account", "show", "--query", "name", "-o", "tsv"],
                              capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            print(f"\n📌 Azure login: ✅ Logged in as subscription: {result.stdout.strip()}")
        else:
            print(f"\n📌 Azure login: ❌ Not logged in — run 'az login' in terminal")
            all_ok = False
    except Exception:
        print(f"\n📌 Azure login: ⚠️ Could not check — run 'az login' in terminal")

print("\n" + "=" * 50)
if all_ok:
    print("🎉 All prerequisites met! Continue to Step 2.")
else:
    print("⚠️  Some prerequisites are missing. Fix them before continuing.")
    if not py_ok:
        print("   👆 Switch the notebook kernel to Python 3.11+ (see instructions above)")
    print("   See PREREQUISITES.md for installation instructions.")

🔍 Checking Prerequisites...

📌 Python (notebook kernel): 3.11.0
   ✅ OK

📌 Azure CLI: ✅ Found

📌 jq: ✅ Found

📌 Git: ✅ Found

📌 Azure login: ✅ Logged in as subscription: ME-MngEnvMCAP338326-robenhai-3

🎉 All prerequisites met! Continue to Step 2.


## Step 2: Deploy Azure Resources

This will deploy all required Azure services using our Bicep template.

**What gets deployed:**
- 🧠 Azure OpenAI (GPT-4.1, GPT-4o-mini, text-embedding-3-large)
- 🔍 Azure AI Search (for RAG in Lab 03)
- 💾 Azure Cosmos DB Serverless (for memory/state in Lab 03)
- 🛡️ Azure AI Content Safety (for guardrails in Lab 05)
- 📦 Storage Account (for documents)

> ⏱️ **This takes about 5-10 minutes.** Go get a coffee! ☕

In [14]:
import subprocess
import json
from pathlib import Path

# Configuration — change these if you want
RESOURCE_GROUP = "rg-agent-platform-labs"
LOCATION = "swedencentral"  # Best availability for all models
BASE_NAME = "agentlabs"

print(f"📦 Resource Group: {RESOURCE_GROUP}")
print(f"🌍 Location: {LOCATION}")
print(f"📛 Base Name: {BASE_NAME}")

# Path to Bicep file
bicep_path = Path("../../infra/main.bicep").resolve()
print(f"📄 Template: {bicep_path}\n")

# Step 1: Create Resource Group
print(f"📦 Creating Resource Group: {RESOURCE_GROUP}...")
rg_result = subprocess.run(
    ["az", "group", "create",
     "--name", RESOURCE_GROUP,
     "--location", LOCATION,
     "--tags", "workshop=agent-platform-labs"],
    capture_output=True, text=True
)
if rg_result.returncode != 0:
    print(f"   ❌ Failed: {rg_result.stderr}")
    raise Exception("Resource group creation failed")
print("   ✅ Resource group created\n")

# Step 2: Deploy Bicep template (same pattern as RAG Workshop)
print("🔧 Deploying Azure resources...")
print("   This may take 5-10 minutes...\n")

deploy_result = subprocess.run(
    ["az", "deployment", "group", "create",
     "--resource-group", RESOURCE_GROUP,
     "--name", "main",
     "--template-file", str(bicep_path),
     "--parameters", f"baseName={BASE_NAME}", f"location={LOCATION}",
     "--output", "json"],
    capture_output=True, text=True,
    timeout=900
)

if deploy_result.returncode != 0:
    print(f"❌ Deployment failed!")
    print(f"Error: {deploy_result.stderr[:2000]}")
    raise Exception("Deployment failed")

print("✅ Deployment completed successfully!")

# Extract outputs
deploy_data = json.loads(deploy_result.stdout)
deployment_outputs = deploy_data.get("properties", {}).get("outputs", {})

print("\n📋 Deployed Resources:")
for key, val_obj in deployment_outputs.items():
    val = val_obj.get("value", "")
    if "Key" in key or "Connection" in key:
        display = val[:20] + "..." if len(val) > 20 else val
    else:
        display = val
    print(f"   • {key}: {display}")

📦 Resource Group: rg-agent-platform-labs
🌍 Location: swedencentral
📛 Base Name: agentlabs
📄 Template: /Users/robenhai/AI Agent Platform/infra/main.bicep

📦 Creating Resource Group: rg-agent-platform-labs...
   ❌ Failed: ERROR: (ResourceGroupBeingDeleted) The resource group 'rg-agent-platform-labs' is in deprovisioning state and cannot perform this operation.
Code: ResourceGroupBeingDeleted
Message: The resource group 'rg-agent-platform-labs' is in deprovisioning state and cannot perform this operation.



Exception: Resource group creation failed

## Step 3: Generate .env File

This extracts all connection strings from the deployment and saves them to `labs/.env`.

In [13]:
from datetime import datetime
from pathlib import Path

# Helper to safely get output values
def get_output(key, default=""):
    return deployment_outputs.get(key, {}).get("value", default)

# Get subscription ID
sub_result = subprocess.run(
    ["az", "account", "show", "--query", "id", "-o", "tsv"],
    capture_output=True, text=True
)
subscription_id = sub_result.stdout.strip()

# Unified AI Services Endpoint & Key
ai_endpoint = get_output("aiServicesEndpoint")
ai_key = get_output("aiServicesKey")

# Generate .env content
env_content = f"""# ===========================================
# AI Agent Platform Labs - Environment Config
# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
# Region: {LOCATION}
# ===========================================
# ⚠️  This file contains secrets. Never commit it to Git!

# Azure Subscription & Resource Group
AZURE_SUBSCRIPTION_ID={subscription_id}
AZURE_RESOURCE_GROUP={RESOURCE_GROUP}
AZURE_LOCATION={LOCATION}

# ── Azure OpenAI (Uses Unified AI Services) ──────────
AZURE_OPENAI_ENDPOINT={ai_endpoint}
AZURE_OPENAI_API_KEY={ai_key}
AZURE_OPENAI_API_VERSION=2024-12-01-preview

# Model deployments
AZURE_OPENAI_DEPLOYMENT_GPT41=gpt-41
AZURE_OPENAI_DEPLOYMENT_GPT4O_MINI=gpt-4o-mini
AZURE_OPENAI_DEPLOYMENT_EMBEDDING=text-embedding-3-large

# ── Azure AI Foundry (Agents, Evaluations, Tracing) ──
AZURE_AI_FOUNDRY_PROJECT={get_output("foundryProjectName")}
AZURE_AI_FOUNDRY_RESOURCE={get_output("aiServicesName")}

# ── Azure AI Search (Lab 03 - RAG) ───────────────────
AZURE_SEARCH_ENDPOINT={get_output("searchServiceEndpoint")}
AZURE_SEARCH_API_KEY={get_output("searchServiceAdminKey")}
AZURE_SEARCH_INDEX_NAME=agent-labs-index

# ── Azure Cosmos DB (Lab 03 - Memory & State) ────────
AZURE_COSMOS_ENDPOINT={get_output("cosmosEndpoint")}
AZURE_COSMOS_KEY={get_output("cosmosKey")}
AZURE_COSMOS_DATABASE={get_output("cosmosDatabaseName")}

# ── Azure AI Content Safety (Lab 05 - Guardrails) ────
AZURE_CONTENT_SAFETY_ENDPOINT={get_output("contentSafetyEndpoint")}
AZURE_CONTENT_SAFETY_KEY={get_output("contentSafetyKey")}

# ── Azure Storage (Lab 03 - Documents) ───────────────
AZURE_STORAGE_CONNECTION_STRING={get_output("storageConnectionString")}
AZURE_STORAGE_CONTAINER_DOCUMENTS=documents
"""

# Write .env file
env_path = Path("../.env").resolve()
env_path.write_text(env_content)

print(f"✅ .env file generated at: {env_path}")
print()
print("📋 Resources configured:")
print(f"   🧠 Azure OpenAI:      {ai_endpoint}")
print(f"   🏗️ AI Foundry Project: {get_output('foundryProjectName')}")
print(f"   🔍 Azure AI Search:    {get_output('searchServiceEndpoint')}")
print(f"   💾 Cosmos DB:          {get_output('cosmosEndpoint')}")
print(f"   🛡️ Content Safety:     {get_output('contentSafetyEndpoint')}")

✅ .env file generated at: /Users/robenhai/AI Agent Platform/labs/.env

📋 Resources configured:
   🧠 Azure OpenAI:      https://ai-agentlabs-fk25xdiuwudo4.cognitiveservices.azure.com/
   🔍 Azure AI Search:    https://search-agentlabs-fk25xdiuwudo4.search.windows.net
   💾 Cosmos DB:          https://cosmos-agentlabs-fk25xdiuwudo4.documents.azure.com:443/
   🛡️ Content Safety:     https://safety-agentlabs-fk25xdiuwudo4.cognitiveservices.azure.com/


## ✅ Setup Complete!

Your environment is ready. Next steps:

1. **Validate:** Open `health-check.ipynb` and run all cells
2. **Start learning:** Open `../lab-01-react-agent/lab.ipynb`

---

> **[→ Run Health Check](health-check.ipynb)** | **[→ Start Lab 01](../lab-01-react-agent/lab.ipynb)**